# 02 — Qualidade dos Dados

Este notebook realiza a **auditoria da qualidade do dataset consolidado** gerado na etapa anterior.

O objetivo é conhecer a estrutura e identificar possíveis problemas antes de qualquer tratamento ou modelagem.

Serão analisados:
- dimensão e estrutura da base;
- tipos de dados;
- valores ausentes;
- registros duplicados;
- cardinalidade das variáveis;
- distribuição das categorias;
- consistência temporal;
- distribuição da variável-alvo `Avaliação Reclamação`.

**Importante:** este notebook é apenas diagnóstico. Nenhum registro será removido e nenhuma variável será recodificada nesta etapa.

In [ ]:
# Importação das bibliotecas
from pathlib import Path
import pandas as pd
import numpy as np


## 1. Configurações iniciais

Ajustes apenas para facilitar a visualização dos resultados no notebook.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Carregamento do dataset consolidado

O arquivo utilizado foi produzido pelo notebook `01_consolidacao_filtro_banco_brasil.ipynb`.

In [ ]:
BASE_DIR = Path(r"C:\UnB\script_md\Base de dados")
INPUT_FILE = BASE_DIR / "processados" / "dataset_banco_brasil_puro.csv"

df = pd.read_csv(
    INPUT_FILE,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print(f"Arquivo carregado: {INPUT_FILE}")
print(f"Dimensão da base: {df.shape[0]:,} linhas x {df.shape[1]} colunas")


## 3. Visão geral da estrutura

Nesta etapa são verificadas as colunas disponíveis e os tipos inferidos pelo pandas.

In [ ]:
# Lista das colunas
df.columns.tolist()


In [ ]:
# Informações estruturais da base
df.info()


## 4. Amostra dos registros

Visualização das primeiras linhas apenas para conferência da estrutura e do conteúdo.

In [ ]:
df.head()


## 5. Auditoria de valores ausentes

Calculamos a quantidade e o percentual de valores ausentes em cada variável.

Neste momento, os valores ausentes são apenas identificados; não serão imputados nem removidos.

In [ ]:
qualidade_nulos = pd.DataFrame({
    "qtd_nulos": df.isna().sum(),
    "pct_nulos": df.isna().mean() * 100
})

qualidade_nulos = qualidade_nulos.sort_values(
    by="pct_nulos",
    ascending=False
)

qualidade_nulos


### Variáveis com pelo menos um valor ausente

In [ ]:
qualidade_nulos[qualidade_nulos["qtd_nulos"] > 0]


## 6. Auditoria de registros duplicados

A duplicidade completa é avaliada considerando todas as colunas do dataset.

A presença de duplicatas não implica automaticamente erro; nesta etapa apenas quantificamos o fenômeno.

In [ ]:
qtd_duplicados = df.duplicated().sum()
pct_duplicados = qtd_duplicados / len(df) * 100

print(f"Registros duplicados: {qtd_duplicados:,}")
print(f"Percentual de duplicados: {pct_duplicados:.2f}%")


### Exemplo de registros duplicados

Caso existam, exibimos uma pequena amostra para inspeção visual.

In [ ]:
df[df.duplicated(keep=False)].head(20)


## 7. Cardinalidade das variáveis

A cardinalidade indica quantos valores distintos existem em cada coluna.

Essa informação será útil posteriormente para identificar:
- variáveis constantes;
- variáveis categóricas de alta cardinalidade;
- possíveis inconsistências de cadastro.

In [ ]:
cardinalidade = pd.DataFrame({
    "n_unicos": df.nunique(dropna=False),
    "dtype": df.dtypes.astype(str)
}).sort_values("n_unicos")

cardinalidade


## 8. Identificação de variáveis constantes

Variáveis com apenas um valor distinto não possuem capacidade discriminante para a modelagem.

In [ ]:
variaveis_constantes = [
    coluna for coluna in df.columns
    if df[coluna].nunique(dropna=False) == 1
]

print("Variáveis constantes:")
variaveis_constantes


## 9. Distribuição das variáveis categóricas

A função abaixo permite inspecionar as categorias mais frequentes de cada variável categórica.

O objetivo é identificar erros de grafia, categorias raras ou padrões inesperados.

In [ ]:
def resumo_categorica(dataframe, coluna, top_n=20):
    """Retorna contagem e percentual das categorias mais frequentes."""
    contagem = dataframe[coluna].value_counts(dropna=False).head(top_n)
    percentual = dataframe[coluna].value_counts(dropna=False, normalize=True).head(top_n) * 100

    return pd.DataFrame({
        "frequencia": contagem,
        "percentual": percentual
    })


### Exemplo: distribuição por UF

In [ ]:
if "UF" in df.columns:
    display(resumo_categorica(df, "UF", top_n=30))


## 10. Auditoria da variável-alvo

A variável-alvo da pesquisa é `Avaliação Reclamação`, com três categorias esperadas:
- `Resolvida`;
- `Não Resolvida`;
- `Não avaliada`.

Nesta etapa verificamos se existem categorias inesperadas e qual é a distribuição observada.

In [ ]:
def localizar_coluna(dataframe, nome_esperado):
    """Localiza uma coluna ignorando espaços duplicados."""
    alvo = " ".join(nome_esperado.split())

    for coluna in dataframe.columns:
        coluna_normalizada = " ".join(str(coluna).split())
        if coluna_normalizada == alvo:
            return coluna

    raise KeyError(f"Coluna '{nome_esperado}' não encontrada.")

COL_TARGET = localizar_coluna(df, "Avaliação Reclamação")

print(f"Coluna-alvo localizada como: {COL_TARGET!r}")


In [ ]:
distribuicao_target = pd.DataFrame({
    "frequencia": df[COL_TARGET].value_counts(dropna=False),
    "percentual": df[COL_TARGET].value_counts(dropna=False, normalize=True) * 100
})

distribuicao_target


### Verificação das categorias esperadas

In [ ]:
categorias_esperadas = {"Resolvida", "Não Resolvida", "Não avaliada"}

categorias_observadas = set(
    df[COL_TARGET]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

categorias_inesperadas = categorias_observadas - categorias_esperadas

print(f"Categorias observadas: {sorted(categorias_observadas)}")
print(f"Categorias inesperadas: {sorted(categorias_inesperadas)}")


## 11. Auditoria temporal

A variável `AAAA_MM`, criada no notebook anterior, representa o período de finalização da reclamação.

Nesta etapa verificamos:
- quantidade de meses disponíveis;
- período inicial e final;
- número de registros por mês;
- possíveis meses ausentes na série.

In [ ]:
if "AAAA_MM" not in df.columns:
    raise KeyError("A coluna 'AAAA_MM' não foi encontrada no dataset.")

periodos = (
    df["AAAA_MM"]
    .dropna()
    .astype(str)
    .sort_values()
)

print(f"Quantidade de períodos distintos: {df['AAAA_MM'].nunique(dropna=True)}")
print(f"Período inicial: {periodos.min()}")
print(f"Período final:   {periodos.max()}")


In [ ]:
registros_por_mes = (
    df["AAAA_MM"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("qtd_registros")
    .to_frame()
)

registros_por_mes


### Verificação de meses ausentes

A sequência observada é comparada a uma sequência mensal completa entre o primeiro e o último período.

In [ ]:
periodos_validos = pd.to_datetime(
    df["AAAA_MM"].dropna().astype(str).str.replace("_", "-", regex=False),
    format="%Y-%m",
    errors="coerce"
).dropna()

inicio = periodos_validos.min().to_period("M")
fim = periodos_validos.max().to_period("M")

sequencia_esperada = pd.period_range(inicio, fim, freq="M")
sequencia_observada = pd.PeriodIndex(periodos_validos.dt.to_period("M").unique())

meses_ausentes = sequencia_esperada.difference(sequencia_observada)

print(f"Quantidade esperada de meses: {len(sequencia_esperada)}")
print(f"Quantidade observada de meses: {len(sequencia_observada)}")
print(f"Meses ausentes: {list(meses_ausentes)}")


## 12. Distribuição da variável-alvo ao longo do tempo

Essa análise permite verificar se a composição das classes varia entre os meses.

Ainda não se trata de uma análise de estabilidade do modelo, mas já ajuda a identificar mudanças importantes na população.

In [ ]:
target_por_mes = pd.crosstab(
    df["AAAA_MM"],
    df[COL_TARGET],
    normalize="index"
) * 100

target_por_mes.round(2)


## 13. Resumo geral da qualidade da base

Consolidamos abaixo alguns indicadores que servirão como primeiro checkpoint antes da preparação dos dados.

In [ ]:
resumo_qualidade = pd.DataFrame({
    "Indicador": [
        "Quantidade de registros",
        "Quantidade de variáveis",
        "Registros duplicados",
        "Variáveis com valores ausentes",
        "Variáveis constantes",
        "Quantidade de meses",
        "Período inicial",
        "Período final"
    ],
    "Valor": [
        f"{len(df):,}",
        df.shape[1],
        f"{qtd_duplicados:,}",
        int((df.isna().sum() > 0).sum()),
        len(variaveis_constantes),
        df["AAAA_MM"].nunique(dropna=True),
        periodos.min(),
        periodos.max()
    ]
})

resumo_qualidade


## Próxima etapa

Os resultados deste notebook deverão ser avaliados antes de qualquer transformação.

Com base nesta auditoria, o próximo notebook poderá tratar:
- definição da amostra supervisionada;
- recodificação da variável-alvo;
- análise de variáveis com risco de *data leakage*;
- tratamento de valores ausentes;
- exclusão justificada de variáveis sem capacidade preditiva;
- preparação para o particionamento temporal.